## Montar carpeta drive

In [ ]:
import os, shutil, random, glob

# Montar Drive
from google.colab import drive
drive.mount('/content/drive')

SRC = "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset_clasificacion"
DST = "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset"

Mounted at /content/drive


## Dividir dataset
- 80% entrenamiento
- 20% validación

In [ ]:

# Crear estructura
for split in ['train', 'val']:
    os.makedirs(f"{DST}/images/{split}", exist_ok=True)
    os.makedirs(f"{DST}/labels/{split}", exist_ok=True)

# Listar todas las imágenes con su .txt
images = sorted(glob.glob(f"{SRC}/*.jpg"))
random.seed(42)
random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_imgs = images[:split_idx]
val_imgs = images[split_idx:]

for img_list, split in [(train_imgs, 'train'), (val_imgs, 'val')]:
    for img_path in img_list:
        base = os.path.splitext(os.path.basename(img_path))[0]
        txt_path = os.path.join(SRC, f"{base}.txt")

        shutil.copy(img_path, f"{DST}/images/{split}/")
        if os.path.exists(txt_path) and os.path.basename(txt_path) != "classes.txt":
            shutil.copy(txt_path, f"{DST}/labels/{split}/")
        else:
            # Crear .txt vacío para negative samples
            open(f"{DST}/labels/{split}/{base}.txt", 'w').close()

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

Train: 146 | Val: 37


## Crear `.yaml`
Crear archivo `.yaml` para pasarselo al modelo

In [ ]:
yaml_content = """
path: /content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset
train: images/train
val: images/val

nc: 2
names: ['bola_roja', 'linea_verde']
"""

with open(f"{DST}/dataset.yaml", 'w') as f:
    f.write(yaml_content)

## Verificar estructura creada

In [ ]:
ls "/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset"

images/  labels/


# Entrenamiento Modelo
(fine-tuning)

## Importar YOLO

In [ ]:
!pip install ultralytics
from ultralytics import YOLO

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Cargar yolov8n

In [ ]:
# Cargar el modelo preentrenado YOLOv8 nano
model = YOLO("yolov8n.pt")

## Entrenamiento

In [ ]:
# Entrenar con transfer learning sobre nuestro dataset
results = model.train(
    data="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/dataset.yaml",
    epochs=100,
    imgsz=320,           # Resolución = la de la cámara del robot
    batch=32,            # lotes
    patience=20,         # Early stopping: para si no mejora en 20 épocas
    project="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs",
    name="tanque_v1",
    cache=True,          # Cachea imágenes en RAM para ir más rápido
    device=0,            # Usar GPU
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=tanque_v1-2, nbs=64, nms=False, opset=None, optimize=Fa

**CAMBIAR RUTA** -> fijarse en el output del entrenamiento.

Cada vez que entrenemos el modelo creará una carpeta nueva (tanque_v1, tanque_v1-2...tanque_v1-n).

Adaptar rutas al modelo entrenado.



Ahora podemos comprobar los resultados:


*   Curvas de entrenamiento
*   Matriz de confusión



In [ ]:
from IPython.display import Image, display

# Mostrar curvas entrenamiento
display(Image('/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/results.png'))

# Mostrar matriz confusion
display(Image('/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/confusion_matrix.png'))

## Validar sobre el conjunto de validación

In [ ]:
# Celda 4: Validar sobre el conjunto de validación
model = YOLO("/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.pt")
metrics = model.val(data="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/dataset.yaml", imgsz=320)
print(f"mAP@0.5: {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.3f}")

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.8±0.3 ms, read: 7.2±2.2 MB/s, size: 11.3 KB)
val: Scanning /content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/dataset/labels/val.cache... 37 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 12.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0it/s 1.5s
                   all         37         58       0.99      0.901      0.952       0.68
             bola_roja         28         30      0.981          1      0.995      0.785
           linea_verde         22         28          1      0.802      0.909      0.576
Speed: 1.7ms preprocess, 14.2ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to /content/runs/detect/val-3
mAP@0.5: 0.952
mAP@0.5:0.95: 0.680

WARNING ⚠️ 
Inference resu

## Comprobar con los vídeos grabados (emulan la situación real)

In [ ]:
# Celda 5: Probar con los vídeos grabados (genera vídeos con las cajas dibujadas)
results = model.predict(
    source="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/videos_prueba/",
    imgsz=320,
    conf=0.4,
    save=True,
    save_txt=True,
    project="/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/inferencia_videos"
)

# Exportar modelo
Una vez listo el modelo, lo exportamos a uno de estos dos formatos
- ONNX
- NCNN

Como solo vamos a hacer inferencias NCNN podría darnos ciertas ventajas.

En el laboratorio vamos a usar ambos formatos a ver cómo se comporta el robot en cada uno.

In [ ]:
# Celda 6: Exportar a ONNX
model = YOLO("/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.pt")
model.export(format="onnx", imgsz=320, simplify=True)

# El archivo best.onnx se guardará junto al best.pt en la carpeta weights/

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.9 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 1.4s, saved as '/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.onnx' (11.6 MB)

Export complete (1.6s)
Results saved to /content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.onnx imgsz=320 
Validate:        yolo val task=detect model=/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanq

'/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.onnx'

In [ ]:
# NCNN es el formato más rápido en ARM (Raspberry Pi)
model.export(format="ncnn", imgsz=320)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.9 MB)

NCNN: starting export with NCNN 1.0.20260114 and PNNX 20260409...
NCNN: export success ✅ 3.2s, saved as '/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best_ncnn_model' (11.6 MB)

Export complete (3.4s)
Results saved to /content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best_ncnn_model
Predict:         yolo predict task=detect model=/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best_ncnn_model imgsz=320 
Validate:        yolo val task=detec

'/content/drive/MyDrive/Colab Notebooks/IAAR/practica3_yolo/runs/tanque_v1-2/weights/best_ncnn_model'